In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q accelerate
!pip install -q beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.8 MB/s eta 0:00:00


In [ ]:
import requests
from bs4 import BeautifulSoup

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [ ]:
url = "https://en.wikipedia.org/wiki/Large_language_model"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

paragraphs = soup.find_all("p")

text = " ".join([p.get_text() for p in paragraphs])

print(len(text))
print(text[:500])

58691
A large language model (LLM) is a language model trained with self-supervised machine learning on a vast amount of data, designed for natural language processing tasks, especially language generation.[1][2] The largest and most capable LLMs are generative pre-trained transformers (GPTs) that provide the core capabilities of modern chatbots. LLMs can be fine-tuned for specific tasks or guided by prompt engineering.[3] These models acquire predictive power regarding syntax, semantics, and ontologi


In [ ]:
chunk_size = 500
chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])

print("Total chunks:", len(chunks))

Total chunks: 118


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = embedding_model.encode(chunks)

print(embeddings.shape)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vector database created")

(118, 384)
Vector database created


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=200
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
def retrieve_context(question):

    question_embedding = embedding_model.encode([question])

    distances, indices = index.search(np.array(question_embedding), k=3)

    results = [chunks[i] for i in indices[0]]

    context = " ".join(results)

    return context

In [ ]:
def ask_rag(question):

    context = retrieve_context(question)

    prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(prompt)[0]["generated_text"]

    return response

In [ ]:
ask_rag("What is a Large Language Model?")

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'\nUse the following context to answer the question.\n\nContext:\nA large language model (LLM) is a language model trained with self-supervised machine learning on a vast amount of data, designed for natural language processing tasks, especially language generation.[1][2] The largest and most capable LLMs are generative pre-trained transformers (GPTs) that provide the core capabilities of modern chatbots. LLMs can be fine-tuned for specific tasks or guided by prompt engineering.[3] These models acquire predictive power regarding syntax, semantics, and ontologi ta, before being fine-tuned.[49]\n Substantial infrastructure is necessary for training the largest models. The tendency towards larger models is visible in the list of large language models. For example, the training of GPT-2 (i.e. a 1.5-billion-parameter model) in 2019 cost $50,000, while training of the PaLM (i.e. a 540-billion-parameter model) in 2022 cost $8 million, and Megatron-Turing NLG 530B (in 2021) cost around $11 mil

In [ ]:
while True:

    question = input("Ask Question: ")

    if question == "exit":
        break

    answer = ask_rag(question)

    print("\nAnswer:\n", answer)

Ask Question: exit


# Fine Tuning

# LoRA – Low-Rank Adaptation

LoRA stands for Low-Rank Adaptation.

It is a technique used to fine-tune large language models efficiently without retraining the entire model.

Large language models like GPT, LLaMA, or Mistral contain billions of parameters.
Training all of them requires huge GPUs, large datasets, and a lot of time.

LoRA solves this problem by freezing the original model weights and adding small trainable layers called adapters.

Instead of modifying the whole model, LoRA trains only a small number of additional parameters.

So during training:



*   The original model remains unchanged
*   Only the small adapter layers are updated





Because of this, LoRA:

*   requires much less GPU memory
*   trains much faster
*   produces very small fine-tuned models

In [ ]:
# To fine-tune large language models efficiently, we install several libraries that help manage datasets, model optimization, and training.
# Libraries used:
# datasets → helps create and manage machine learning datasets
# peft → used for Parameter Efficient Fine-Tuning (PEFT) techniques such as LoRA
# bitsandbytes → allows models to be loaded in 4-bit or 8-bit precision, reducing GPU memory usage
# trl → provides tools like SFTTrainer for supervised fine-tuning of language models

# These libraries make it possible to fine-tune large models even on limited GPUs like Google Colab.

!pip install -q datasets peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 20.2 MB/s eta 0:00:00


In [ ]:
# Before fine-tuning, we need a pretrained base model.
# In this notebook we use: TinyLlama (1.1B parameters)
# This model is smaller than many modern LLMs, making it suitable for experimentation and educational purposes.
# To reduce GPU memory usage, we load the model in 4-bit precision using BitsAndBytes.
# Quantization reduces the memory required to store model weights while maintaining acceptable performance.

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Since the model is loaded in 4-bit quantized format, its weights cannot be directly updated during training.
# Instead, we use LoRA (Low-Rank Adaptation).
# LoRA works by:
# 1. Freezing the original model parameters
# 2. Adding small trainable adapter layers
# 3. Updating only those small layers during training
# This significantly reduces training cost and GPU memory usage.

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# Fine-tuning requires a dataset in Instruction → Input → Output format.
# Since our notebook already scraped a Wikipedia article and divided it into chunks, we can use those chunks to generate training examples.
# Each example teaches the model to explain a concept related to Large Language Models.

training_data = []

for chunk in chunks[:50]:   # keep small for faster training
    training_data.append({
        "instruction": "Explain the following concept about Large Language Models.",
        "input": chunk,
        "output": chunk
    })

print("Total training samples:", len(training_data))

Total training samples: 50


In [ ]:
# Machine learning frameworks require structured datasets.
# The HuggingFace Dataset format allows:
  # efficient data loading
  # dataset transformations
  # compatibility with training frameworks
# So we convert our Python list into a Dataset object.

from datasets import Dataset

dataset = Dataset.from_list(training_data)

In [ ]:
# Language models are trained using text prompts, not structured JSON.
# Therefore, we convert each dataset entry into a prompt format:
  #Instructions:
  #Input:
  #Response:

def format_prompt(example):

    prompt = f"""
Instruction: {example['instruction']}

Input:
{example['input']}

Response:
{example['output']}
"""

    return {"text": prompt}

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
# Language models cannot process raw text directly.
# They operate on tokens, which are numerical representations of words or subwords.
# Tokenization converts text into token IDs that the model can understand.

def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
# Before training, we specify parameters that control the training process.
# Important parameters:
  # batch size → number of samples processed at once
  # epochs → number of times the model sees the dataset
  # logging steps → frequency of training logs

from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./tinyllama-finetuned",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10
)

In [ ]:
# We use SFTTrainer (Supervised Fine-Tuning Trainer) from the TRL library.
# This trainer simplifies the fine-tuning process by handling:
  # gradient updates
  # batching
  # training loops
  # optimization
# During training, only the LoRA adapter parameters are updated.

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args
)

trainer.train()

Truncating train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,3.165877
20,2.312277


TrainOutput(global_step=25, training_loss=2.5808074569702146, metrics={'train_runtime': 20.5637, 'train_samples_per_second': 2.431, 'train_steps_per_second': 1.216, 'total_flos': 159074117222400.0, 'train_loss': 2.5808074569702146})

In [ ]:
# After training, we save the updated model so it can be reused later.
# This avoids retraining the model every time we want to use it.

trainer.model.save_pretrained("tinyllama-finetuned")
tokenizer.save_pretrained("tinyllama-finetuned")

('tinyllama-finetuned/tokenizer_config.json',
 'tinyllama-finetuned/chat_template.jinja',
 'tinyllama-finetuned/tokenizer.json')

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="tinyllama-finetuned",
    tokenizer=tokenizer,
    max_new_tokens=200
)

print(generator("Explain Large Language Models")[0]["generated_text"])


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain Large Language Models (LLMs) and how they work, including their benefits and limitations. Discuss how LLMs have been used in various industries, such as marketing and finance. Provide examples of how LLMs have been integrated into language learning platforms, including how they improve accuracy and engagement. Additionally, provide insights into the future of LLMs and their potential for advancing language learning and business communication.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "tinyllama-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200
)

In [ ]:
def ask_model(question):

    prompt = f"""
Instruction: Answer the question.

Question: {question}

Response:
"""

    response = generator(prompt)[0]["generated_text"]

    return response

In [ ]:
print(ask_model("What is a Large Language Model?"))

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Instruction: Answer the question.

Question: What is a Large Language Model?

Response:

A Large Language Model (LLM) is a type of language model that can process vast amounts of text. They are used for tasks such as text generation, language translation, and natural language processing. LLMs are often trained on large corpora of text, including text from multiple languages and diverse domains. They are often used in applications such as chatbots, language translation, and text generation.

In summary, a Large Language Model is a type of language model that processes large amounts of text and is commonly used in applications such as chatbots, language translation, and natural language processing.


In [ ]:
while True:

    question = input("Ask Question: ")

    if question.lower() == "exit":
        break

    answer = ask_model(question)

    print("\nAnswer:\n", answer)

Ask Question: exit
